In [1]:
import pandas as pd
import json
import random

# Set random seed for reproducibility
random.seed(42)

# Read the CSV file
csv_path = '../../results/opengrep_results.csv'
df = pd.read_csv(csv_path)

# Display basic info
print(f"Total rows: {len(df)}")
print(f"\nLanguages in CSV: {df['language'].unique()}")
print(f"\nConversations per language:")
print(df.groupby('language')['conversation_hash'].nunique())

Total rows: 7781

Languages in CSV: ['c' 'csharp' 'java' 'javascript' 'php' 'python']

Conversations per language:
language
c              595
csharp         120
java           244
javascript     674
php             49
python        1464
Name: conversation_hash, dtype: int64


In [45]:
# Define the languages we want to process
target_languages = ["python", "c", "java", "javascript", "php", "csharp"]

# Get unique conversation hashes per language and sample 200 random ones
sampled_conversations = {}

for lang in target_languages:
    # Filter by language (case-insensitive comparison)
    lang_df = df[df["language"].str.lower() == lang.lower()]

    # Get unique conversation hashes
    grouped = lang_df.groupby("conversation_hash")
    print(f"{lang}: {len(grouped)} unique conversations")
    # Sample 200 random conversations (or all if less than 200)
    n_samples = min(250, lang_df["conversation_hash"].nunique())
    sampled_hashes = lang_df["conversation_hash"].drop_duplicates().sample(n_samples)
    sampled_df = lang_df[lang_df["conversation_hash"].isin(sampled_hashes)]
    result = (
        sampled_df.groupby("conversation_hash")
        .apply(
            lambda g: {
                "error_messages": g["error_message"].tolist(),
                "user_prompt": None,
                "llm_response": None
            }
        )
        .to_dict()
    )
    sampled_conversations[lang] = result
    print(f"  Sampled: {len(result)} conversations")

print("\nSampled conversations per language:")
for lang, hashes in sampled_conversations.items():
    print(f"  {lang}: {len(hashes)} conversations")

python: 1464 unique conversations
  Sampled: 250 conversations
c: 595 unique conversations
  Sampled: 250 conversations
java: 244 unique conversations
  Sampled: 244 conversations
javascript: 674 unique conversations
  Sampled: 250 conversations
php: 49 unique conversations
  Sampled: 49 conversations
csharp: 120 unique conversations
  Sampled: 120 conversations

Sampled conversations per language:
  python: 250 conversations
  c: 250 conversations
  java: 244 conversations
  javascript: 250 conversations
  php: 49 conversations
  csharp: 120 conversations


/tmp/ipykernel_106511/4067411378.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_106511/4067411378.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_106511/4067411378.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation.

In [46]:
sampled_conversations

{'python': {'007275f88060ad984743c14e09cf0254': {'error_messages': ["Detected Flask app with debug=True. Do not deploy to production with this flag enabled as it will leak sensitive information. Instead, consider using Flask configuration variables or setting 'debug' using system environment variables."],
   'user_prompt': None,
   'llm_response': None},
  '0098bb9111bdc2fe5e51e15e8f322c0f': {'error_messages': ['Avoid using `pickle`, which is known to lead to code execution vulnerabilities. When unpickling, the serialized data could be manipulated to run arbitrary code. Instead, consider serializing the relevant data as JSON or a similar text-based serialization format.',
    'The Python documentation recommends using `defusedxml` instead of `xml` because the native Python `xml` library is vulnerable to XML External Entity (XXE) attacks. These attacks can leak confidential data and "XML bombs" can cause denial of service.',
    'The Python documentation recommends using `defusedxml` in

## Load WildChat Dataset to Get User Prompts

Now we'll load the WildChat-1M dataset from HuggingFace to extract the first user prompt from each conversation.

In [15]:
from datasets import load_dataset
from tqdm import tqdm

# Load the full dataset (not streaming) and convert to pandas
wildchat_dataset = load_dataset("allenai/WildChat-1M", split="train")

In [47]:
# Extract the first user message from each conversation using filter
print("\nExtracting first user prompts from conversations...")

# Convert needed hashes to a set for faster lookup
all_needed_hashes = set()
for lang in target_languages:
    all_needed_hashes.update(sampled_conversations[lang])

print(f"Looking for {len(all_needed_hashes)} unique conversation hashes")

def check_hash(example):
    """Check if conversation hash is in our needed set"""
    return example['conversation_hash'] in all_needed_hashes

def extract_first_user_message(example):
    """Extract the first user message from a conversation that has at least 20 characters"""
    conversation = example['conversation']
    if isinstance(conversation, list) and len(conversation) > 0:
        for turn in conversation:
            if isinstance(turn, dict) and turn.get('role') == 'user':
                content = turn.get('content', '')
                # Check if message has at least 20 characters
                if len(content) >= 20:
                    example['first_user_msg'] = content
                    return example
    # No valid user message found (either no user turns or all < 20 chars)
    example['first_user_msg'] = None
    return example

# Filter the dataset to only needed conversations and extract first user message
print("Filtering and extracting with multiprocessing...")
from multiprocessing import cpu_count

filtered_dataset = wildchat_dataset.filter(check_hash, num_proc=cpu_count())
print(f"Filtered down to {len(filtered_dataset)} conversations")

# Extract first user messages
processed_dataset = filtered_dataset.map(extract_first_user_message, num_proc=cpu_count())

# Create dictionary mapping conversation_hash to first_user_msg
conversation_prompts = {
    row['conversation_hash']: row['first_user_msg'] 
    for row in processed_dataset 
    if row['first_user_msg'] is not None
}

print(f"\nFound {len(conversation_prompts)} out of {len(all_needed_hashes)} needed conversations")
print(f"Skipped {len(all_needed_hashes) - len(conversation_prompts)} conversations (no user messages >= 20 chars)")
print(f"Created lookup dictionary with {len(conversation_prompts)} conversation prompts")


Extracting first user prompts from conversations...
Looking for 1158 unique conversation hashes
Filtering and extracting with multiprocessing...


Filter (num_proc=20): 100%|██████████| 837989/837989 [00:40<00:00, 20507.22 examples/s]


Filtered down to 1158 conversations


Map (num_proc=20): 100%|██████████| 1158/1158 [00:03<00:00, 322.98 examples/s]



Found 1121 out of 1158 needed conversations
Skipped 37 conversations (no user messages >= 20 chars)
Created lookup dictionary with 1121 conversation prompts


In [48]:

# Update the result dictionary with the user prompts and remove conversations without valid prompts
print("\nUpdating result dictionary with user prompts...")

for lang in tqdm(target_languages, desc="Processing languages"):
    updated = 0
    to_remove = []
    
    for conv_hash in sampled_conversations[lang]:
        if conv_hash in conversation_prompts:
            sampled_conversations[lang][conv_hash]['user_prompt'] = conversation_prompts[conv_hash]
            updated += 1
        else:
            # Mark for removal if no valid user prompt found
            to_remove.append(conv_hash)
    
    # Remove conversations without valid user prompts
    for conv_hash in to_remove:
        del sampled_conversations[lang][conv_hash]
    
    print(f"  {lang}: Kept {updated}/{updated + len(to_remove)} conversations (Removed {len(to_remove)} without valid prompts)")

print("\nDone! All conversations with valid user prompts have been updated.")
print(f"Total conversations kept: {sum(len(v) for v in sampled_conversations.values())}")


Updating result dictionary with user prompts...


Processing languages: 100%|██████████| 6/6 [00:00<00:00, 3836.25it/s]

  python: Kept 248/250 conversations (Removed 2 without valid prompts)
  c: Kept 238/250 conversations (Removed 12 without valid prompts)
  java: Kept 235/244 conversations (Removed 9 without valid prompts)
  javascript: Kept 237/250 conversations (Removed 13 without valid prompts)
  php: Kept 49/49 conversations (Removed 0 without valid prompts)
  csharp: Kept 119/120 conversations (Removed 1 without valid prompts)

Done! All conversations with valid user prompts have been updated.
Total conversations kept: 1126


In [49]:
# Display a sample of the result
print("Sample of the result dictionary:\n")
for lang in target_languages:  # Show first 2 languages
    print(f"\n{lang}:")
    sample_hashes = list(sampled_conversations[lang].keys())[:5]  # Show first 3 conversations
    for conv_hash in sample_hashes:
        print(f"  {conv_hash}: {sampled_conversations[lang][conv_hash]}")
    if len(sampled_conversations[lang]) > 3:
        print(f"  ... and {len(sampled_conversations[lang]) - 3} more conversations")

Sample of the result dictionary:


python:
  007275f88060ad984743c14e09cf0254: {'error_messages': ["Detected Flask app with debug=True. Do not deploy to production with this flag enabled as it will leak sensitive information. Instead, consider using Flask configuration variables or setting 'debug' using system environment variables."], 'user_prompt': '如何使用gensim 3.7.0版本调用腾讯开源的词向量包，获取每一个词的向量数据，请用python实现，链接在这“https://ai.tencent.com/ailab/nlp/en/download.html”，并且对外暴露restful API ，接口入参是词语，出参是词语对应的向量数组。请用mvc分层架构设计代码', 'llm_response': None}
  0098bb9111bdc2fe5e51e15e8f322c0f: {'error_messages': ['Avoid using `pickle`, which is known to lead to code execution vulnerabilities. When unpickling, the serialized data could be manipulated to run arbitrary code. Instead, consider serializing the relevant data as JSON or a similar text-based serialization format.', 'The Python documentation recommends using `defusedxml` instead of `xml` because the native Python `xml` library is vulnerable to XML Ext

In [50]:
# Save the result to a JSON file
output_path = '/home/regularpooria/Projects/WildCode/results/sampled_conversations.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(sampled_conversations, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {output_path}")
print(f"\nTotal conversations: {sum(len(v) for v in sampled_conversations.values())}")

Results saved to: /home/regularpooria/Projects/WildCode/results/sampled_conversations.json

Total conversations: 1126
